In [1]:
import json
import random
import re
import shutil
import textwrap
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
TRAIN_RATIO = 0.7
VALID_RATIO = 0.15
TEST_RATIO = 0.15

PROJECT_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = DATA_ROOT / "merged_yolo_grouped"
METADATA_DIR = OUTPUT_ROOT / "metadata"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

assert abs(TRAIN_RATIO + VALID_RATIO + TEST_RATIO - 1.0) < 1e-9

OUTPUT_ROOT, METADATA_DIR

(PosixPath('/home/thanhmay/workspace/xai-driven-saliency-loss/data/merged_yolo_grouped'),
 PosixPath('/home/thanhmay/workspace/xai-driven-saliency-loss/data/merged_yolo_grouped/metadata'))

In [2]:
def canonical_name(image_info: dict) -> str:
    return image_info.get("extra", {}).get("name", image_info["file_name"])


def stem_without_suffix(image_name: str) -> str:
    return Path(image_name).stem


def group_key_from_stem(stem: str) -> str:
    return re.sub(r"_crop_\d+$", "", stem)


def discover_annotation_files(data_root: Path) -> list[Path]:
    return sorted(data_root.glob("**/_annotations.coco.json"))


def parse_annotation_source(ann_path: Path) -> dict:
    relative = ann_path.relative_to(DATA_ROOT)
    parts = relative.parts
    dataset = parts[0]
    original_split = parts[1]
    field = parts[2] if len(parts) > 3 else "mixed"
    orientation = dataset.replace("Data_", "")
    return {
        "dataset": dataset,
        "original_split": original_split,
        "field": field,
        "orientation": orientation,
    }


def build_file_index(root_dir: Path) -> dict[str, Path]:
    file_index: dict[str, Path] = {}
    for path in root_dir.rglob("*"):
        if path.is_file() and path.name != "_annotations.coco.json":
            file_index.setdefault(path.name, path)
    return file_index


def yolo_line(category_id_zero_based: int, bbox: list[float], width: int, height: int) -> str:
    x_min, y_min, box_w, box_h = bbox
    x_center = (x_min + box_w / 2) / width
    y_center = (y_min + box_h / 2) / height
    norm_w = box_w / width
    norm_h = box_h / height
    return (
        f"{category_id_zero_based} "
        f"{x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}"
    )

In [3]:
GLOBAL_FILE_INDEX = build_file_index(DATA_ROOT)
annotation_files = discover_annotation_files(DATA_ROOT)
annotation_files

[PosixPath('/home/thanhmay/workspace/xai-driven-saliency-loss/data/Data_side/test/Bright_Field/_annotations.coco.json'),
 PosixPath('/home/thanhmay/workspace/xai-driven-saliency-loss/data/Data_side/test/Dark_Field/_annotations.coco.json'),
 PosixPath('/home/thanhmay/workspace/xai-driven-saliency-loss/data/Data_side/train/_annotations.coco.json'),
 PosixPath('/home/thanhmay/workspace/xai-driven-saliency-loss/data/Data_side/valid/_annotations.coco.json'),
 PosixPath('/home/thanhmay/workspace/xai-driven-saliency-loss/data/Data_top/train/_annotations.coco.json')]

In [4]:
category_id_to_name = None
all_records: list[dict] = []

for ann_path in annotation_files:
    source_info = parse_annotation_source(ann_path)
    coco = json.loads(ann_path.read_text())
    if category_id_to_name is None:
        category_id_to_name = {item["id"]: item["name"] for item in coco["categories"]}

    annotations_by_image: dict[int, list[dict]] = defaultdict(list)
    for ann in coco["annotations"]:
        annotations_by_image[ann["image_id"]].append(ann)

    for image_info in coco["images"]:
        image_name = canonical_name(image_info)
        stem = stem_without_suffix(image_name)
        group_key = group_key_from_stem(stem)
        image_path = GLOBAL_FILE_INDEX.get(image_info["file_name"])
        resolved_field = source_info["field"]
        if image_path is not None:
            relative_parts = set(image_path.relative_to(PROJECT_ROOT).parts)
            if "Bright_Field" in relative_parts:
                resolved_field = "Bright_Field"
            elif "Dark_Field" in relative_parts:
                resolved_field = "Dark_Field"

        all_records.append(
            {
                **{**source_info, "field": resolved_field},
                "ann_path": str(ann_path),
                "image_id": image_info["id"],
                "image_name": image_name,
                "file_name": image_info["file_name"],
                "stem": stem,
                "group_key": group_key,
                "image_key": f"{source_info['orientation']}::{stem}",
                "image_path": str(image_path) if image_path is not None else "",
                "width": int(image_info["width"]),
                "height": int(image_info["height"]),
                "annotations": annotations_by_image.get(image_info["id"], []),
                "annotation_count": len(annotations_by_image.get(image_info["id"], [])),
            }
        )

raw_df = pd.DataFrame(all_records)
raw_df.shape

(19728, 16)

In [5]:
existing_mask = raw_df["image_path"].map(lambda value: bool(value) and Path(value).is_file())
missing_files = raw_df.loc[~existing_mask, ["file_name", "image_name", "ann_path"]].copy()
usable_df = raw_df.loc[existing_mask].copy().reset_index(drop=True)

dedup_df = (
    usable_df.sort_values(["image_key", "original_split", "ann_path"])
    .drop_duplicates(subset=["image_key"], keep="first")
    .reset_index(drop=True)
)

duplicate_rows = usable_df[usable_df.duplicated(subset=["image_key"], keep="first")].copy()

print(f"Raw images referenced by COCO files: {len(raw_df):,}")
print(f"Images physically found in workspace: {len(usable_df):,}")
print(f"Missing image files skipped: {len(missing_files):,}")
print(f"Unique images kept after deduplication: {len(dedup_df):,}")
print(f"Duplicate image entries removed: {len(duplicate_rows):,}")
print(f"Unique source groups after deduplication: {dedup_df['group_key'].nunique():,}")

Raw images referenced by COCO files: 19,728
Images physically found in workspace: 11,091
Missing image files skipped: 8,637
Unique images kept after deduplication: 10,367
Duplicate image entries removed: 724
Unique source groups after deduplication: 1,241


In [6]:
category_names = [category_id_to_name[idx] for idx in sorted(category_id_to_name)]
category_names

['drill', 'Broken', 'Chipped', 'Scratched', 'Severe_Rust', 'Tip_Wear']

In [7]:
def collect_group_stats(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for group_key, group_df in df.groupby("group_key"):
        label_counter = Counter()
        for anns in group_df["annotations"]:
            for ann in anns:
                label_counter[category_id_to_name[ann["category_id"]]] += 1

        positive_labels = sorted(label_counter)
        dominant_label = label_counter.most_common(1)[0][0] if label_counter else "background"

        rows.append(
            {
                "group_key": group_key,
                "image_count": int(len(group_df)),
                "annotation_count": int(sum(group_df["annotation_count"])),
                "positive_label_count": int(sum(label_counter.values())),
                "positive_classes": positive_labels,
                "signature": "|".join(positive_labels) if positive_labels else "background",
                "dominant_label": dominant_label,
                "orientations": sorted(group_df["orientation"].unique()),
                "fields": sorted(group_df["field"].unique()),
            }
        )

    return pd.DataFrame(rows).sort_values("group_key").reset_index(drop=True)


group_df = collect_group_stats(dedup_df)
group_df.head()

,group_key,image_count,annotation_count,positive_label_count,positive_classes,signature,dominant_label,orientations,fields
0,S100_Image__2025-09-10__11-46-17_bright_1,2,4,4,[Tip_Wear],Tip_Wear,Tip_Wear,"[side, top]",[Bright_Field]
1,S100_Image__2025-09-10__11-47-01_dark_1,2,4,4,[Tip_Wear],Tip_Wear,Tip_Wear,"[side, top]",[Bright_Field]
2,S101_Image__2025-09-10__12-16-33_dark_2,14,24,24,"[Broken, Chipped, Severe_Rust]",Broken|Chipped|Severe_Rust,Broken,"[side, top]",[Dark_Field]
3,S101_Image__2025-09-10__12-16-33_dark_2_crop_4...,2,4,4,"[Broken, Chipped]",Broken|Chipped,Broken,"[side, top]",[Dark_Field]
4,S101_Image__2025-09-10__12-21-26_bright_2,16,18,18,"[Broken, Chipped]",Broken|Chipped,Broken,"[side, top]",[Bright_Field]


In [8]:
def choose_stratify_labels(group_table: pd.DataFrame) -> pd.Series | None:
    label_counts = group_table["dominant_label"].value_counts()
    if (label_counts < 2).any():
        return None
    return group_table["dominant_label"]


def split_groups(group_table: pd.DataFrame) -> pd.DataFrame:
    stratify_labels = choose_stratify_labels(group_table)
    train_groups, temp_groups = train_test_split(
        group_table,
        test_size=(1.0 - TRAIN_RATIO),
        random_state=RANDOM_SEED,
        shuffle=True,
        stratify=stratify_labels,
    )

    temp_ratio = VALID_RATIO + TEST_RATIO
    valid_fraction_of_temp = VALID_RATIO / temp_ratio

    stratify_temp = choose_stratify_labels(temp_groups)
    valid_groups, test_groups = train_test_split(
        temp_groups,
        test_size=(1.0 - valid_fraction_of_temp),
        random_state=RANDOM_SEED,
        shuffle=True,
        stratify=stratify_temp,
    )

    split_map = {}
    split_map.update({group_key: "train" for group_key in train_groups["group_key"]})
    split_map.update({group_key: "valid" for group_key in valid_groups["group_key"]})
    split_map.update({group_key: "test" for group_key in test_groups["group_key"]})

    result = group_table.copy()
    result["split"] = result["group_key"].map(split_map)
    return result


group_split_df = split_groups(group_df)
group_split_df["split"].value_counts()

split
train    868
test     187
valid    186
Name: count, dtype: int64

In [9]:
split_lookup = group_split_df.set_index("group_key")["split"].to_dict()
merged_df = dedup_df.copy()
merged_df["split"] = merged_df["group_key"].map(split_lookup)
merged_df["is_positive"] = merged_df["annotation_count"] > 0

leakage_check = (
    merged_df.groupby("group_key")["split"].nunique().gt(1).sum()
)
assert leakage_check == 0, f"Found {leakage_check} leaked groups across splits."

merged_df[["split", "orientation", "field"]].value_counts().rename("image_count").reset_index().head(20)

,split,orientation,field,image_count
0,train,side,Bright_Field,2195
1,train,side,Dark_Field,1952
2,train,top,Bright_Field,1656
3,train,top,Dark_Field,1399
4,valid,side,Bright_Field,553
5,test,side,Dark_Field,456
6,valid,top,Bright_Field,454
7,test,side,Bright_Field,417
8,test,top,Bright_Field,345
9,test,top,Dark_Field,344


In [10]:
def reset_output_dir(output_root: Path) -> None:
    if output_root.exists():
        shutil.rmtree(output_root)

    for split in ["train", "valid", "test"]:
        (output_root / "images" / split).mkdir(parents=True, exist_ok=True)
        (output_root / "labels" / split).mkdir(parents=True, exist_ok=True)

    (output_root / "metadata").mkdir(parents=True, exist_ok=True)


def output_stem(row: pd.Series) -> str:
    return f"{row['orientation']}__{row['stem']}"


reset_output_dir(OUTPUT_ROOT)

In [11]:
exported_rows: list[dict] = []
category_id_to_zero_based = {
    original_id: new_idx for new_idx, original_id in enumerate(sorted(category_id_to_name))
}

for _, row in merged_df.iterrows():
    source_path = Path(row["image_path"])
    destination_stem = output_stem(row)
    destination_image = OUTPUT_ROOT / "images" / row["split"] / f"{destination_stem}{source_path.suffix.lower()}"
    destination_label = OUTPUT_ROOT / "labels" / row["split"] / f"{destination_stem}.txt"

    shutil.copy2(source_path, destination_image)

    yolo_rows = [
        yolo_line(
            category_id_zero_based=category_id_to_zero_based[ann["category_id"]],
            bbox=ann["bbox"],
            width=row["width"],
            height=row["height"],
        )
        for ann in row["annotations"]
    ]

    destination_label.write_text("\n".join(yolo_rows))

    class_names = sorted({category_id_to_name[ann["category_id"]] for ann in row["annotations"]})
    exported_rows.append(
        {
            "split": row["split"],
            "group_key": row["group_key"],
            "image_key": row["image_key"],
            "dataset": row["dataset"],
            "original_split": row["original_split"],
            "field": row["field"],
            "orientation": row["orientation"],
            "stem": row["stem"],
            "image_name": row["image_name"],
            "source_path": str(source_path.relative_to(PROJECT_ROOT)),
            "output_image": str(destination_image.relative_to(PROJECT_ROOT)),
            "output_label": str(destination_label.relative_to(PROJECT_ROOT)),
            "width": int(row["width"]),
            "height": int(row["height"]),
            "annotation_count": int(row["annotation_count"]),
            "is_positive": bool(row["annotation_count"] > 0),
            "class_names": "|".join(class_names),
        }
    )

In [12]:
exported_df = pd.DataFrame(exported_rows)

bbox_rows: list[dict] = []
for _, row in merged_df.iterrows():
    for ann in row["annotations"]:
        bbox_rows.append(
            {
                "split": row["split"],
                "group_key": row["group_key"],
                "image_key": row["image_key"],
                "orientation": row["orientation"],
                "field": row["field"],
                "dataset": row["dataset"],
                "class_name": category_id_to_name[ann["category_id"]],
                "bbox_x": ann["bbox"][0],
                "bbox_y": ann["bbox"][1],
                "bbox_w": ann["bbox"][2],
                "bbox_h": ann["bbox"][3],
                "bbox_area": ann["bbox"][2] * ann["bbox"][3],
                "image_width": row["width"],
                "image_height": row["height"],
                "bbox_area_ratio": (ann["bbox"][2] * ann["bbox"][3]) / (row["width"] * row["height"]),
            }
        )

bbox_df = pd.DataFrame(bbox_rows)

split_summary = (
    exported_df.groupby("split")
    .agg(
        image_count=("image_key", "count"),
        group_count=("group_key", "nunique"),
        positive_images=("is_positive", "sum"),
        negative_images=("is_positive", lambda values: int((~values).sum())),
    )
    .reset_index()
)

class_summary = (
    bbox_df.groupby(["split", "class_name"]).size().rename("annotation_count").reset_index()
    if not bbox_df.empty
    else pd.DataFrame(columns=["split", "class_name", "annotation_count"])
)

source_notes = []
unlabeled_folders = sorted(
    str(path.relative_to(PROJECT_ROOT))
    for path in DATA_ROOT.glob("**/*")
    if path.is_dir() and "Data_top/valid" in str(path) and not list(path.glob("_annotations.coco.json"))
)
if unlabeled_folders:
    source_notes.append(
        "Skipped directories without COCO annotation files, including Data_top/valid images."
    )

summary = {
    "random_seed": RANDOM_SEED,
    "ratios": {"train": TRAIN_RATIO, "valid": VALID_RATIO, "test": TEST_RATIO},
    "annotation_files": [str(path.relative_to(PROJECT_ROOT)) for path in annotation_files],
    "raw_images": int(len(raw_df)),
    "found_images": int(len(usable_df)),
    "missing_files_skipped": int(len(missing_files)),
    "unique_images_after_dedup": int(len(dedup_df)),
    "removed_duplicates": int(len(duplicate_rows)),
    "unique_groups": int(dedup_df["group_key"].nunique()),
    "images_without_annotations": int((exported_df["annotation_count"] == 0).sum()),
    "split_counts": split_summary.to_dict(orient="records"),
    "class_counts": (
        bbox_df.groupby("class_name").size().rename("annotation_count").reset_index().to_dict(orient="records")
        if not bbox_df.empty
        else []
    ),
    "notes": source_notes,
}

exported_df.to_csv(METADATA_DIR / "image_metadata.csv", index=False)
bbox_df.to_csv(METADATA_DIR / "bbox_metadata.csv", index=False)
group_split_df.to_csv(METADATA_DIR / "group_metadata.csv", index=False)
duplicate_rows.drop(columns=["annotations"]).to_csv(METADATA_DIR / "removed_duplicates.csv", index=False)
missing_files.to_csv(METADATA_DIR / "missing_files.csv", index=False)
(METADATA_DIR / "summary.json").write_text(json.dumps(summary, indent=2))

dataset_yaml = textwrap.dedent(
    f"""
    path: {OUTPUT_ROOT.relative_to(PROJECT_ROOT)}
    train: images/train
    val: images/valid
    test: images/test
    names:
    """
).strip("\n")

for idx, name in enumerate(category_names):
    dataset_yaml += f"\n  {idx}: {name}"

(OUTPUT_ROOT / "dataset.yaml").write_text(dataset_yaml + "\n")

summary

{'random_seed': 42,
 'ratios': {'train': 0.7, 'valid': 0.15, 'test': 0.15},
 'annotation_files': ['data/Data_side/test/Bright_Field/_annotations.coco.json',
  'data/Data_side/test/Dark_Field/_annotations.coco.json',
  'data/Data_side/train/_annotations.coco.json',
  'data/Data_side/valid/_annotations.coco.json',
  'data/Data_top/train/_annotations.coco.json'],
 'raw_images': 19728,
 'found_images': 11091,
 'missing_files_skipped': 8637,
 'unique_images_after_dedup': 10367,
 'removed_duplicates': 724,
 'unique_groups': 1241,
 'images_without_annotations': 1984,
 'split_counts': [{'split': 'test',
   'image_count': 1562,
   'group_count': 187,
   'positive_images': 1227,
   'negative_images': 335},
  {'split': 'train',
   'image_count': 7202,
   'group_count': 868,
   'positive_images': 5842,
   'negative_images': 1360},
  {'split': 'valid',
   'image_count': 1603,
   'group_count': 186,
   'positive_images': 1314,
   'negative_images': 289}],
 'class_counts': [{'class_name': 'Broken', '